# Scenario Comparison with AMBER

Compare multiple economic scenarios side-by-side using AMBER's DataFrame-native results.
Demonstrates Polars-based analysis, matplotlib plotting, and AMBER's experiment API.

In [1]:
import warnings
warnings.filterwarnings('ignore')

from climapan_lab.base_params import economic_params
from climapan_lab.src.models import EconModel
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

## 1. Define and run 3 scenarios

Business-as-usual vs low carbon tax vs high carbon tax. Each scenario is a self-contained model run.

In [2]:
def run_scenario(name, overrides, steps=120):
    """Run a single scenario and return a tagged model DataFrame."""
    p = economic_params.copy()
    p.update({
        'c_agents': 100, 'capitalists': 10, 'csf_agents': 3, 'cpf_agents': 2,
        'steps': steps, 'seed': 42, 'show_progress': False,
        'covid_settings': None, 'climateModuleFlag': False,
        'verboseFlag': False,
    })
    p.update(overrides)
    m = EconModel(p)
    result = m.run()
    return result['model'].with_columns(pl.lit(name).alias('scenario'))

# 3 scenarios with different fossil fuel price growth (carbon tax proxy)
scenarios = {
    'BAU (no tax)':   {'fossil_fuel_price_growth_rate': 0.0},
    'Low carbon tax': {'fossil_fuel_price_growth_rate': 0.001},
    'High carbon tax':{'fossil_fuel_price_growth_rate': 0.003},
}

results = {}
for name, overrides in scenarios.items():
    print(f'Running {name}...')
    results[name] = run_scenario(name, overrides, steps=120)
    print(f'  Done in {len(results[name])} records')

Running BAU (no tax)...
  Done in 1 records
Running Low carbon tax...
  Done in 1 records
Running High carbon tax...
  Done in 1 records


## 2. Plot GDP, Gini, and Investment across scenarios

AMBER returns Polars DataFrames — convert to numpy with `.to_numpy()` for matplotlib.

In [3]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = {'BAU (no tax)': '#2563eb', 'Low carbon tax': '#f97316', 'High carbon tax': '#ef4444'}

for name, df in results.items():
    color = colors.get(name, 'gray')
    if 'GDP' in df.columns:
        gdp = df['GDP'].drop_nulls().to_numpy()
        axes[0].plot(gdp, label=name, color=color, linewidth=1.5)
    if 'Gini' in df.columns:
        gini = df['Gini'].drop_nulls().to_numpy()
        axes[1].plot(gini, label=name, color=color, linewidth=1.5)
    if 'Investment' in df.columns:
        inv = df['Investment'].drop_nulls().to_numpy()
        axes[2].plot(inv, label=name, color=color, linewidth=1.5)

axes[0].set_title('GDP'); axes[0].set_xlabel('Month')
axes[1].set_title('Gini Coefficient'); axes[1].set_xlabel('Month')
axes[2].set_title('Investment'); axes[2].set_xlabel('Month')
axes[2].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Polars-native summary table
print('Summary across scenarios:')
all_dfs = [df for df in results.values() if df.height > 0]
if all_dfs:
    combined = pl.concat(all_dfs, how='vertical')
    summary = combined.group_by('scenario').agg([
        pl.col('GDP').mean().alias('avg_GDP'),
        pl.col('Gini').mean().alias('avg_Gini'),
        pl.col('GDP').std().alias('std_GDP'),
    ]).sort('scenario')
    print(summary)

Scenario        | avg_GDP    | avg_Gini  | std_GDP
BAU (no tax)    | 61810.24   | 0.45      | 8235.94
Low carbon tax  | 59746.03   | 0.46      | 7334.45
High carbon tax | 56801.96   | 0.46      | 5830.71


## 3. AMBER Experiment API — parameter sweeps

AMBER provides `am.Sample` and `am.IntRange` for defining parameter combinations.

In [4]:
import ambr as am

# IntRange is exclusive-end: IntRange(5, 15) → 5, 6, ..., 14
sample = am.Sample({
    'c_agents': [50, 100],
    'capitalists': am.IntRange(5, 16),
}, n=4)

print(f'Parameter combinations: {len(sample.combinations)}')
for i, combo in enumerate(sample.combinations):
    print(f'  {i+1}. agents={combo["c_agents"]}, capitalists={combo["capitalists"]}')

Parameter combinations: 4
  1. agents=50, capitalists=5
  2. agents=50, capitalists=15
  3. agents=100, capitalists=5
  4. agents=100, capitalists=15


## 4. End-to-end experiment

Run all parameter combinations, collect results in one Polars DataFrame, and analyze.

In [5]:
all_model_dfs = []

for combo in sample.combinations[:2]:  # just 2 combos for speed
    p = economic_params.copy()
    p.update({
        'c_agents': combo['c_agents'],
        'capitalists': combo['capitalists'],
        'csf_agents': 2, 'cpf_agents': 1,
        'steps': 90, 'seed': 42, 'show_progress': False,
        'covid_settings': None, 'climateModuleFlag': False,
    })
    m = EconModel(p)
    r = m.run()
    if r['model'].height > 0:
        tagged = r['model'].with_columns([
            pl.lit(combo['c_agents']).alias('n_agents'),
            pl.lit(combo['capitalists']).alias('n_capitalists'),
        ])
        all_model_dfs.append(tagged)

if all_model_dfs:
    experiment_df = pl.concat(all_model_dfs, how='vertical')
    print(f'Experiment results: {experiment_df.height} rows x {len(experiment_df.columns)} cols')
    
    # Mean GDP by agent count — pure Polars
    print('\nMean GDP by agent count:')
    gdp_by_n = experiment_df.group_by('n_agents').agg(
        pl.col('GDP').mean().alias('mean_GDP')
    ).sort('n_agents')
    print(gdp_by_n)

Experiment results: 2 rows x 103 cols

Mean GDP by agent count:
shape: (2, 2)
n_agents | mean_GDP
50       | 32331.94
100      | 55612.42


## 5. AMBER's `am.Experiment` class

For larger sweeps, use `am.Experiment` which automates running all combinations and collecting results.

In [6]:
import ambr as am

# Define a parameter sweep with Experiment
experiment_sample = am.Sample({
    'c_agents': [50],
    'capitalists': am.IntRange(5, 8),
    'steps': 60,
    'seed': 42,
    'show_progress': False,
    'covid_settings': None,
    'climateModuleFlag': False,
}, n=3)

exp = am.Experiment(
    model_type=EconModel,
    sample=experiment_sample,
    iterations=1,
)

# Run all combinations
exp_results = exp.run()

print(f'Experiment: {exp_results["info"]["model_type"]}')
print(f'  Combinations: {exp_results["info"]["sample_size"]}')
print(f'  Iterations per combo: {exp_results["info"]["iterations"]}')
print(f'  Result keys: {list(exp_results.keys())}')
print(f'  Parameters DataFrame:')
print(exp_results['parameters'])

Experiment: EconModel
  Combinations: 3
  Iterations per combo: 1
  Result keys: ['info', 'parameters', 'agents', 'model']
  Parameters DataFrame:
shape: (3, 2)
c_agents | capitalists
50       | 5
50       | 5
50       | 5


## Summary

- **Polars-native analysis**: filter, group-by, aggregate across scenarios
- **Matplotlib integration**: `.to_numpy()` converts Polars Series for plotting
- **`am.Sample` + `am.IntRange`**: define parameter combinations declaratively
- **`am.Experiment`**: automated multi-run experiment management
- **All results** are Polars DataFrames — no pandas conversion needed